# Fine Tuning and Prompt Dataset Creation

This notebook fine tunes an OpenAI GPT model, using synthetically created user prompts (from GPT-4) from the the original https://github.com/yukiar/CEFR-SP train data set.  


In [1]:
import os

os.environ['OPENAI_API_KEY'] = "redacted"


In [2]:
# Make the directory to download the original CEFR-SP train and test datasets
!mkdir -p dataset

# Download (from my personal S3 bucket instead of directly from their Github) the original CEFR-SP train and test
!wget -O dataset/data_train.txt https://my-shared-downloads.s3.us-east-1.amazonaws.com/CEFR-SP_Wikiauto_train.txt
!wget -O dataset/data_test.txt https://my-shared-downloads.s3.us-east-1.amazonaws.com/CEFR-SP_Wikiauto_test.txt

# Create a directory to store a JSON verson of the dataset, augmented with synthetic user prompts
!mkdir -p augmented

!pip install openai

--2025-04-14 00:37:19--  https://my-shared-downloads.s3.us-east-1.amazonaws.com/CEFR-SP_Wikiauto_train.txt
Resolving my-shared-downloads.s3.us-east-1.amazonaws.com (my-shared-downloads.s3.us-east-1.amazonaws.com)... 52.217.114.170, 52.217.198.58, 52.217.130.2, ...
Connecting to my-shared-downloads.s3.us-east-1.amazonaws.com (my-shared-downloads.s3.us-east-1.amazonaws.com)|52.217.114.170|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 565805 (553K) [text/plain]
Saving to: ‘dataset/data_train.txt’

dataset/data_train. 100%[===================>] 552.54K  1.75MB/s    in 0.3s    

2025-04-14 00:37:20 (1.75 MB/s) - ‘dataset/data_train.txt’ saved [565805/565805]

--2025-04-14 00:37:20--  https://my-shared-downloads.s3.us-east-1.amazonaws.com/CEFR-SP_Wikiauto_test.txt
Resolving my-shared-downloads.s3.us-east-1.amazonaws.com (my-shared-downloads.s3.us-east-1.amazonaws.com)... 52.217.114.170, 52.217.198.58, 52.217.130.2, ...
Connecting to my-shared-downloads.s3.us-east-

In [15]:

def read_raw_dataset(file_path):
    """
    Reads a tab-delimited dataset file where each line has:
    Sentence \t Label by annotator A \t Label by annotator B

    Returns a list of dictionaries:
    [
        {"sentence": "...", "label_a": 1, "label_b": 2},
        ...
    ]
    """
    dataset = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            parts = line.strip().split('\t')
            if len(parts) != 3:
                print(f"Warning: Line {line_num} malformed: {line.strip()}")
                continue
            sentence, label_a, label_b = parts
            try:
                dataset.append({
                    "sentence": sentence,
                    "label_a": int(label_a),
                    "label_b": int(label_b)
                })
            except ValueError:
                print(f"Warning: Line {line_num} has non-integer labels: {label_a}, {label_b}")
                continue
    return dataset

import openai
import os

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def generate_synthetic_prompt(sentence, model="gpt-4", temperature=0.7):
    """
    Given a sentence (completion), generate a synthetic user prompt that could have
    prompted a chatbot to produce it.

    Args:
        sentence (str): The sentence from the dataset (assistant response).
        model (str): The OpenAI chat model to use.
        temperature (float): Sampling temperature.

    Returns:
        str: A synthetic user prompt.
    """
    prompt = (
        f"You are an assistant in a chat. Given the assistant's reply, "
        f"generate a synthetic user prompt that could have led to that response.\n\n"
        f"Assistant's reply: \"{sentence}\"\n"
        f"User's prompt:"
    )

    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": "You generate realistic user prompts for chatbot fine-tuning."},
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content.strip()

import time
from typing import List, Dict

def add_synthetic_prompts(dataset: List[Dict], delay: float = 0.1) -> None:
    """
    Updates each dict in the dataset with a synthetic user prompt.

    Each dict must already contain a "sentence" key.
    A new key "prompt" will be added with the generated synthetic prompt.

    Args:
        dataset (List[Dict]): List of dataset items with "sentence" keys.
        delay (float): Optional delay between API calls to avoid rate limits.
    """
    for i, example in enumerate(dataset):
        sentence = example.get("sentence")
        if not sentence:
            continue

        try:
            prompt = generate_synthetic_prompt(sentence)
            example["prompt"] = prompt
        except Exception as e:
            print(f"⚠️ Error generating prompt for item {i}: {e}")
            example["prompt"] = None  # Or skip entirely depending on your needs
            continue

        time.sleep(delay)

import json

def save_augmented_dataset_as_jsonl(dataset: List[Dict], filepath: str) -> None:
    """
    Saves the augmented dataset (list of dicts) to a JSONL file.

    Args:
        dataset (List[Dict]): The dataset to save.
        filepath (str): The path to the JSONL file to write.
    """
    with open(filepath, "w", encoding="utf-8") as f:
        for example in dataset:
            json.dump(example, f, ensure_ascii=False)
            f.write("\n")



In [20]:
train = read_raw_dataset("dataset/data_train.txt")
test = read_raw_dataset("dataset/data_test.txt")

add_synthetic_prompts(train)
add_synthetic_prompts(test)

# The synthetic prompt generation for train and test combined was expensive
# time-wise (around 2.5 hours) so we save it to disk, and then permanently
# store it (manually via the web based AWS S3 console) into my-shared-downloads bucket
save_augmented_dataset_as_jsonl(train, "augmented/train.jsonl")
save_augmented_dataset_as_jsonl(test, "augmented/test.jsonl")


In [8]:
# Now we can just download the augmented train and test datasets, complete with
# the GPT-4 generated synthetic user prompts.
!mkdir -p augmented

!wget -O augmented/train.jsonl https://my-shared-downloads.s3.us-east-1.amazonaws.com/train.jsonl
!wget -O augmented/test.jsonl https://my-shared-downloads.s3.us-east-1.amazonaws.com/test.jsonl



--2025-04-14 00:39:32--  https://my-shared-downloads.s3.us-east-1.amazonaws.com/train.jsonl
Resolving my-shared-downloads.s3.us-east-1.amazonaws.com (my-shared-downloads.s3.us-east-1.amazonaws.com)... 16.182.65.114, 16.15.177.11, 16.15.178.110, ...
Connecting to my-shared-downloads.s3.us-east-1.amazonaws.com (my-shared-downloads.s3.us-east-1.amazonaws.com)|16.182.65.114|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1300652 (1.2M) [binary/octet-stream]
Saving to: ‘augmented/train.jsonl’

augmented/train.jso 100%[===================>]   1.24M  2.88MB/s    in 0.4s    

2025-04-14 00:39:33 (2.88 MB/s) - ‘augmented/train.jsonl’ saved [1300652/1300652]

--2025-04-14 00:39:33--  https://my-shared-downloads.s3.us-east-1.amazonaws.com/test.jsonl
Resolving my-shared-downloads.s3.us-east-1.amazonaws.com (my-shared-downloads.s3.us-east-1.amazonaws.com)... 16.182.65.114, 16.15.177.11, 16.15.178.110, ...
Connecting to my-shared-downloads.s3.us-east-1.amazonaws.com (my-sha

In [11]:
import json
from typing import List, Dict

def load_augmented_dataset_from_jsonl(filepath):
    """
    Loads an augmented dataset (list of dicts) from a JSONL file.

    Args:
        filepath (str): The path to the JSONL file to read.

    Returns:
        List[Dict]: The loaded dataset.
    """
    dataset = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            dataset.append(json.loads(line))
    return dataset

train = load_augmented_dataset_from_jsonl("augmented/train.jsonl")
test = load_augmented_dataset_from_jsonl("augmented/test.jsonl")


# Fine Tuning

In [12]:
import io

def convert_to_multiturn_finetune_format(system_prompt_fn, dataset):
    """
    Converts a dataset with 'prompt', 'sentence', 'label_a', and 'label_b' into
    OpenAI multi-turn fine-tuning format with a system message describing the CEFR level.

    We need to vary the system message based on whether it is actually for fine tuning or
    for direct usage as few-show prompting.  We do this via the system_prompt_fn
    argument, which is a function that given the level_str as an argument will
    produce the relevante system message.

    Returns:
        List[Dict]: Each item has a "messages" key with a list of role-content pairs.
    """
    level_map = {
        1: "A1",
        2: "A2",
        3: "B1",
        4: "B2",
        5: "C1",
        6: "C2"
    }

    examples = []

    for i, example in enumerate(dataset):
        sentence = example.get("sentence")
        prompt = example.get("prompt")
        label_a = example.get("label_a")
        label_b = example.get("label_b")

        if sentence is None or prompt is None or label_a is None or label_b is None:
            print(f"⚠️ Skipping incomplete example at index {i}")
            continue

        try:
            # Choose the higher of the two CEFR levels
            level_num = max(label_a, label_b)
            level_str = level_map.get(level_num, "UNKNOWN")

            system_msg = system_prompt_fn(level_str)

            examples.append({
                "messages": [
                    {"role": "system", "content": system_msg},
                    {"role": "user", "content": prompt},
                    {"role": "assistant", "content": sentence}
                ]
            })

        except Exception as e:
            print(f"Error building example {i}: {e}")
            continue

    return examples

import random

def build_examples_by_level(dataset):
    examples_by_level = {
        "1": [], "2": [], "3": [],
        "4": [], "5": [], "6": []
    }
    for example in dataset:
        sentence = example.get("sentence")
        label_a = example.get("label_a")
        label_b = example.get("label_b")

        level_num = max(label_a, label_b)
        examples_by_level[str(level_num)].append(sentence)
    return examples_by_level

def fine_tuned_system_msg_builder(level_str):
    return f"You are a chat bot assistant which restricts your output language to CEFR level {level_str}."

def n_shot_system_msg_builder(n, examples_by_level):
    """
    Returns a lambda that takes a CEFR level_str (e.g. 'B1') and returns a system message
    with that level and N example sentences for each level from 1 to 6.

    examples_by_level: dict of {"1": [...], "2": [...], ..., "6": [...]}
    """
    cefr_map = {
        "1": "A1", "2": "A2", "3": "B1",
        "4": "B2", "5": "C1", "6": "C2"
    }

    def make_system_msg(level_str):
        base_msg = f"You are a chat bot assistant which restricts your output language to CEFR level {level_str}.\n"

        if n == 0:
            return base_msg

        examples_section = []

        for level_num_str in map(str, range(1, 7)):
            examples = examples_by_level.get(level_num_str, [])
            if examples:
                sampled = random.sample(examples, min(n, len(examples)))
                cefr_label = cefr_map[level_num_str]
                for i, ex in enumerate(sampled):
                    examples_section.append(f"Example ({cefr_label}): {ex}")

        return base_msg + "\n" + "\n".join(examples_section)

    return lambda level_str: make_system_msg(level_str)


# From: https://cookbook.openai.com/examples/chat_finetuning_data_prep
def validate_dataset(dataset):
  # Format error checks
    format_errors = {}

    for ex in dataset:
        if not isinstance(ex, dict):
            format_errors["data_type"] += 1
            continue

        messages = ex.get("messages", None)
        if not messages:
            format_errors["missing_messages_list"] += 1
            continue

        for message in messages:
            if "role" not in message or "content" not in message:
                format_errors["message_missing_key"] += 1

            if any(k not in ("role", "content", "name", "function_call", "weight") for k in message):
                format_errors["message_unrecognized_key"] += 1

            if message.get("role", None) not in ("system", "user", "assistant", "function"):
                format_errors["unrecognized_role"] += 1

            content = message.get("content", None)
            function_call = message.get("function_call", None)

            if (not content and not function_call) or not isinstance(content, str):
                format_errors["missing_content"] += 1

        if not any(message.get("role", None) == "assistant" for message in messages):
            format_errors["example_missing_assistant_message"] += 1

    if format_errors:
        print("Found errors:")
        for k, v in format_errors.items():
            print(f"{k}: {v}")
    else:
        print("No errors found")

def upload_finetune_data_from_memory(data: List[Dict], filename: str = "finetuning.jsonl") -> str:
    """
    Uploads a list of dicts (fine-tuning format) to OpenAI as a .jsonl file without saving to disk.

    Args:
        data (List[Dict]): The fine-tuning training data.
        filename (str): A pretend filename for metadata purposes.

    Returns:
        str: The uploaded file ID.
    """
    buffer = io.BytesIO()

    for example in data:
        line = json.dumps(example, ensure_ascii=False)
        buffer.write(line.encode("utf-8") + b"\n")

    buffer.seek(0)
    response = client.files.create(
        file=(filename, buffer),
        purpose="fine-tune"
    )

    print("✅ Uploaded in-memory file:", response.filename)
    print("📁 File ID:", response.id)
    return response.id

In [13]:

examples_by_level = build_examples_by_level(train)

# Create all of necessary message builder functions
zero_shot_fn = n_shot_system_msg_builder(0, examples_by_level)
one_shot_fn = n_shot_system_msg_builder(1, examples_by_level)
two_shot_fn = n_shot_system_msg_builder(2, examples_by_level)
three_shot_fn = n_shot_system_msg_builder(3, examples_by_level)

# Create our complete OpenAI compliant fine tuning dataset
fine_tune_dataset = convert_to_multiturn_finetune_format(fine_tuned_system_msg_builder, train)

# Validate it using the code found in the OpenAI docs
validate_dataset(fine_tune_dataset)



No errors found


In [48]:
# Create the fine tuning job
# Update the contents into OpenAI as fine tuning file
file = upload_finetune_data_from_memory(fine_tune_dataset, "finetuning.jsonl")

client.fine_tuning.jobs.create(
    training_file=file,
    model="gpt-4o-mini-2024-07-18"
)

client.fine_tuning.jobs.list(limit=10)



SyncCursorPage[FineTuningJob](data=[FineTuningJob(id='ftjob-9BvKvsS7pYyNy7s9W9PvGiKD', created_at=1744494175, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-3wt9bYu0Y95oK423mTjFGFu0', result_files=[], seed=1611990223, status='validating_files', trained_tokens=None, training_file='file-RrncoDB8X73M74j9qRSUnU', validation_file=None, estimated_finish=None, integrations=[], metadata=None, method=Method(dpo=None, supervised=MethodSupervised(hyperparameters=MethodSupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto')), type='supervised'), user_provided_suffix=None), FineTuningJob(id='ftjob-UTTus58pedo6TWPuORXrHlKx', created_at=1744493135, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyp

In [68]:

# By executing this over and over, we can monitor the status of the fine tuning
# job.  At completion, I received an email with the name of the fine tuned model
client.fine_tuning.jobs.list_events(fine_tuning_job_id="ftjob-9BvKvsS7pYyNy7s9W9PvGiKD", limit=10)


SyncCursorPage[FineTuningJobEvent](data=[FineTuningJobEvent(id='ftevent-WrIahzaAjkTa3W9UVArYToeT', created_at=1744496321, level='info', message='The job has successfully completed', object='fine_tuning.job.event', data={}, type='message'), FineTuningJobEvent(id='ftevent-pKjNNLz7O563bBM3xd1PMD0V', created_at=1744496314, level='info', message='New fine-tuned model created', object='fine_tuning.job.event', data={}, type='message'), FineTuningJobEvent(id='ftevent-ovStE664PqrD2PIJO9aMVfAJ', created_at=1744496314, level='info', message='Checkpoint created at step 1090', object='fine_tuning.job.event', data={}, type='message'), FineTuningJobEvent(id='ftevent-hJ1bLWbLrqDbos0V5DEVQBq7', created_at=1744496314, level='info', message='Checkpoint created at step 545', object='fine_tuning.job.event', data={}, type='message'), FineTuningJobEvent(id='ftevent-zXUT9e7UKzlPjS3dYynJNyZJ', created_at=1744496298, level='info', message='Step 1634/1634: training loss=0.84', object='fine_tuning.job.event', dat

In [17]:
!mkdir -p test

In [18]:
# Now let's generate all of the datasets that we need for testing the prompting
# few show approach.  This gives us datasets in four variation: zero examples,
# one, two and three exampes (from each class)

one_shot_fn = n_shot_system_msg_builder(1, examples_by_level)
two_shot_fn = n_shot_system_msg_builder(2, examples_by_level)
three_shot_fn = n_shot_system_msg_builder(3, examples_by_level)

zero_examples_dataset = convert_to_multiturn_finetune_format(fine_tuned_system_msg_builder, test)
one_examples_dataset = convert_to_multiturn_finetune_format(one_shot_fn, test)
two_examples_dataset = convert_to_multiturn_finetune_format(two_shot_fn, test)
three_examples_dataset = convert_to_multiturn_finetune_format(three_shot_fn, test)

# Save them all in the notebook, and then I manually downloaded them and manually
# uploaded them into publicly available S3 bucket
save_augmented_dataset_as_jsonl(zero_examples_dataset, "test/zero.jsonl")
save_augmented_dataset_as_jsonl(one_examples_dataset, "test/one.jsonl")
save_augmented_dataset_as_jsonl(two_examples_dataset, "test/two.jsonl")
save_augmented_dataset_as_jsonl(three_examples_dataset, "test/three.jsonl")

